In [22]:
import pandas as pd
import getpass
from huggingface_hub import notebook_login
import os

from datasets import load_dataset
from fireworks.client import Fireworks
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM
import json
from openai import OpenAI
import re


In [3]:
os.environ["FIREWORKS_API_KEY"] = getpass.getpass("fireworks api:")
client = Fireworks(api_key=os.environ["FIREWORKS_API_KEY"])

In [4]:
notebook_login()

In [5]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("openai api:")

openai_client = OpenAI(
    # This is the default and can be omitted
    api_key=os.environ.get("OPENAI_API_KEY"),
)

In [6]:
model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

/home/markt/.local/lib/python3.11/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [7]:
instruction = """
          you are an expert translator between English and Danish

          #user will provide you sentences in English 
          #translate users english sentence to Danish.
          #you can only use users english sentence
          
  """


def translate_english_to_danish(english_samples_csv_file, model):
    list_of_danish_sentences = list()
    df = pd.read_csv(english_samples_csv_file)
    for i, row in enumerate(df.iterrows()):
        danish_sentence = client.chat.completions.create(
            model=model,
            messages=[
              {"role": "system", "content": instruction},
              {"role": "user", "content": row[1]['English']}
            ],
        )
        response = danish_sentence.choices[0].message.content
        list_of_danish_sentences.append(response)    
    return list_of_danish_sentences


generated_senetences_english_to_danish = translate_english_to_danish('./english-danish-openai.csv','accounts/fireworks/models/llama-v3-8b-instruct')


In [8]:
generated_senetences_english_to_danish

['Katten sad på vinduesillsen og så på fuglene udenfor.',
 'Hun baking en smagfuld chokoladekage til hendes veninde fødselsdag.',
 'Nyt park i bycentret er blevet en populær destination for familier.',
 'Han fik hurtigt indset, at at lære et nyt sprog kræver tålmodighed og øving.',
 'Den gamle borg på bakkerne tilbyder en smuk udsigt over den omgivende landsbygård.',
 'De valgte at tilbringe deres ferie ved at udforske de fjernlejsende øer i Stillehavet.',
 'Fysikeren forenede en revolutionerende opdagelse, der kan ændre fremtidens medicin.',
 'Børnene var energiske til at starte deres første skoldag efter sommerferie.',
 'Hun fuldførte marathonen trods de krævende vejrforhold.',
 'Det innovative start-up skal skabe bæredygtige løsninger for byliv.',
 'Jeg grandfætter fortalte ham historier om de gamle dage, da landsbyen var meget mindre.',
 'Luften af nybrygget kaffe fylde rummet.',
 'Under stjernekysten lovede de at forblive venner i evighed.',
 'Selvom regnen var tung, fortsatte fes

In [38]:
instruction = """
        you are an expert translator between English and Danish

        #user will only give you samples of a sentence translated from English to Danish#
        # Give the translation a score on a scale from one to ten#
        #format should be something like 'score 1'#
        #give me an example how I can improve the instruction in order to improve score#
        #Think through your reasoning step-by-step and write the score in a new line at the end#
        score as int and placed as the last part#   
        #always write the score as '\nScore ' + the score - dont add other things!
              

          
  """


# instruction = """
#         you are an expert translator between English and Danish

#         #user will only give you samples of a sentence translated from English to Danish#
#         # Give the translation a score on a scale from one to ten#
#         #format should only be your score          
#   """


def evaluate_danish_sentences(english_danish_open_csv, evaluation_sentences):
    scores = list()
    df = pd.read_csv(english_danish_open_csv)
    df["Danish_llama_3"] = evaluation_sentences

    for i, row in enumerate(df.iterrows()): 
        
        response = openai_client.chat.completions.create(
            messages=[
                {"role": "system", "content": instruction},
                {"role": "user", "content": f'''English:{row[1]['English']} 
                                                Danish:{row[1]['Danish_llama_3']}'''}
            ],
            model="gpt-4o",

        )
        try:
            response = response.choices[0].message.content
            print(response + '\n\n' + '----------' + '\n\n')
            score = int(re.search("\d?\d", re.search("Score?:?\s\d?\d|\*?\*?\*?score?:?\*?\*\s+\d?\d", response)[0])[0])            
            scores.append(score)
        except json.JSONDecodeError as jde:
            continue

    return sum(scores) / len(scores)


llama_8b_avg_score = evaluate_danish_sentences("./english-danish-openai.csv", generated_senetences_english_to_danish)


To evaluate the translation accurately, let's break it down:

1. **Katten** - Correct translation for "The cat".
2. **sad** - Correct translation for "sat".
3. **på** - Correct preposition for "on".
4. **vinduesillsen** - This word contains a spelling error. The correct spelling in Danish is "vindueskarmen".
5. **og så** - Correct translation for "watching".
6. **på fuglene** - Correct translation for "the birds".
7. **udenfor** - Correct translation for "outside".

The primary issue lies in the spelling of "vinduesillsen". With the corrected translation, it would read:
"Katten sad på vindueskarmen og så på fuglene udenfor."

Score: 8

----------


The translation "Hun baking en smagfuld chokoladekage til hendes veninde fødselsdag." has errors in both grammar and word choice. "Hun baking" should be "Hun bagte" for the correct past tense form, and "hendes veninde fødselsdag" should be "hendes venindes fødselsdag" for the correct possessive form.

The correct translation would be: "Hun b

TypeError: 'NoneType' object is not subscriptable

In [ ]:
# print(f"Llama3 8B: {round(llama_8b_avg_score, 2)}")
print(llama_8b_avg_score)